<a href="https://colab.research.google.com/github/sadeelfaleh/dentex/blob/main/dentex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q huggingface_hub ultralytics
from huggingface_hub import list_repo_files
for f in list_repo_files("ibrahimhamamci/DENTEX", repo_type="dataset"):
    print(f)

import torch
print("GPU متاح؟", torch.cuda.is_available())
if torch.cuda.is_available():
    print("الكرت:", torch.cuda.get_device_name(0))

.gitattributes
DENTEX/test_data.zip
DENTEX/training_data.zip
DENTEX/validation_data.zip
DENTEX/validation_triple.json
README.md
figures/data.png
figures/dentex.jpg
figures/output.png
GPU متاح؟ True
الكرت: Tesla T4


In [5]:
from huggingface_hub import hf_hub_download
import shutil, os

for f in ["DENTEX/validation_triple.json",
          "DENTEX/validation_data.zip",
          "DENTEX/training_data.zip"]:
    p = hf_hub_download(repo_id="ibrahimhamamci/DENTEX", repo_type="dataset",
                        filename=f, local_dir="/content/raw")
    print("تم:", p)

shutil.unpack_archive("/content/raw/DENTEX/validation_data.zip", "/content/raw/val_x")
shutil.unpack_archive("/content/raw/DENTEX/training_data.zip", "/content/raw/train_x")
print("تم فك الضغط")

# نحذف ملفات zip لتوفير مساحة القرص
os.remove("/content/raw/DENTEX/validation_data.zip")
os.remove("/content/raw/DENTEX/training_data.zip")

تم: /content/raw/DENTEX/validation_triple.json
تم: /content/raw/DENTEX/validation_data.zip


DENTEX/training_data.zip: reconstructing file:   0%|          |  0.00B / 10.9GB            

DENTEX/training_data.zip: downloading bytes:           |  0.00B            

تم: /content/raw/DENTEX/training_data.zip
تم فك الضغط


In [7]:
import json, shutil
from pathlib import Path
from collections import Counter, defaultdict

RAW = Path("/content/raw")
OUT = Path("/content/dentex_yolo")
EXPECTED = ['Impacted', 'Caries', 'Periapical Lesion', 'Deep Caries']
CATS_KEY, LABEL_KEY = "categories_3", "category_id_3"

def find_xrays(base):
    for p in Path(base).rglob("xrays"):
        if p.parent.name.replace("-", "_") == "quadrant_enumeration_disease":
            return p

train_json = next((RAW / "train_x").rglob("train_quadrant_enumeration_disease.json"))
val_json   = RAW / "DENTEX" / "validation_triple.json"
train_imgs = find_xrays(RAW / "train_x")
val_imgs   = find_xrays(RAW / "val_x")
print("train:", train_json, "|", train_imgs)
print("val:  ", val_json, "|", val_imgs)

def convert(json_path, imgs_dir, split):
    data = json.loads(json_path.read_text(encoding="utf-8"))
    cats = sorted(data[CATS_KEY], key=lambda c: c["id"])
    names = [c["name"] for c in cats]
    assert names == EXPECTED, f"ترتيب الكلاسات مختلف: {names}"
    id_map = {c["id"]: i for i, c in enumerate(cats)}

    img_out = OUT / "images" / split
    lbl_out = OUT / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    by_img = defaultdict(list)
    for a in data["annotations"]:
        by_img[a["image_id"]].append(a)

    missing, clipped, boxes, empty = [], 0, 0, 0
    cnt = Counter()
    for img in data["images"]:
        fname = Path(img["file_name"]).name
        W, H = img["width"], img["height"]
        src = imgs_dir / fname
        if not src.exists():
            missing.append(fname)
            continue
        shutil.copy2(src, img_out / fname)
        lines = []
        for a in by_img.get(img["id"], []):
            x, y, w, h = a["bbox"]
            vals = [(x + w / 2) / W, (y + h / 2) / H, w / W, h / H]
            if any(v < 0 or v > 1 for v in vals):
                clipped += 1
            xc, yc, nw, nh = (min(1.0, max(0.0, v)) for v in vals)
            c = id_map[a[LABEL_KEY]]
            lines.append(f"{c} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
            cnt[names[c]] += 1
            boxes += 1
        if not lines:
            empty += 1
        (lbl_out / (Path(fname).stem + ".txt")).write_text("\n".join(lines), encoding="utf-8")

    print(f"\n[{split}] صور: {len(list(img_out.glob('*.png')))} | مربعات: {boxes} | "
          f"قص: {clipped} | ناقصة: {len(missing)} | بدون مربعات: {empty}")
    print("توزيع الكلاسات:", dict(cnt))
    return names

names = convert(train_json, train_imgs, "train")
convert(val_json, val_imgs, "val")

yaml_text = f"""path: {OUT.as_posix()}
train: images/train
val: images/val

nc: {len(names)}
names: {names}
"""
(OUT / "dentex.yaml").write_text(yaml_text, encoding="utf-8")
print("\nتم إنشاء dentex.yaml")

train: /content/raw/train_x/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json | /content/raw/train_x/training_data/quadrant-enumeration-disease/xrays
val:   /content/raw/DENTEX/validation_triple.json | /content/raw/val_x/validation_data/quadrant_enumeration_disease/xrays

[train] صور: 705 | مربعات: 3529 | قص: 0 | ناقصة: 0 | بدون مربعات: 27
توزيع الكلاسات: {'Impacted': 604, 'Caries': 2189, 'Periapical Lesion': 158, 'Deep Caries': 578}

[val] صور: 50 | مربعات: 182 | قص: 0 | ناقصة: 0 | بدون مربعات: 4
توزيع الكلاسات: {'Impacted': 40, 'Caries': 101, 'Periapical Lesion': 9, 'Deep Caries': 32}

تم إنشاء dentex.yaml


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.make_archive("/content/drive/MyDrive/dentex_yolo_upload", "zip", "/content", "dentex_yolo")
print("تم الحفظ على Drive")

Mounted at /content/drive
تم الحفظ على Drive


In [9]:
import ultralytics, torch
from ultralytics import YOLO

print("ultralytics:", ultralytics.__version__, "| torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolov8s.pt")   # أوزان COCO الجاهزة
model.train(
    data="/content/dentex_yolo/dentex.yaml",
    epochs=3,
    imgsz=1280,
    batch=8,
    seed=0,
    workers=2,
    project="/content/drive/MyDrive/dentex_runs",
    name="smoke_test",
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics: 8.4.194 | torch: 2.11.0+cu130
GPU: Tesla T4
Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dentex_yolo/dentex.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, f

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fbfde5f9cd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [11]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data="/content/dentex_yolo/dentex.yaml",
    epochs=100,
    patience=20,
    imgsz=1280,
    batch=8,
    seed=0,
    workers=2,
    project="/content/drive/MyDrive/dentex_runs",
    name="baseline_v1",
)

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dentex_yolo/dentex.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_v1-2, nbs=64, nms=None, opset=None, optimize=Fa

KeyboardInterrupt: 

In [12]:
import os
d = "/content/dentex_yolo"
print("الداتا؟", os.path.exists(d),
      "| train:", len(os.listdir(f"{d}/images/train")) if os.path.exists(d) else 0,
      "| val:", len(os.listdir(f"{d}/images/val")) if os.path.exists(d) else 0)

w = "/content/drive/MyDrive/dentex_runs/baseline_v1/weights/best.pt"
print("best.pt على Drive؟", os.path.exists(w))

try:
    import ultralytics
    print("ultralytics:", ultralytics.__version__)
except Exception as e:
    print("ultralytics ناقصة:", e)

الداتا؟ True | train: 705 | val: 50
best.pt على Drive؟ False
ultralytics: 8.4.194


In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [20]:
import os, torch
d = "/content/dentex_yolo"
print("الداتا؟", os.path.exists(d),
      "| train:", len(os.listdir(f"{d}/images/train")) if os.path.exists(d) else 0,
      "| val:", len(os.listdir(f"{d}/images/val")) if os.path.exists(d) else 0)
print("GPU:", torch.cuda.is_available())

الداتا؟ True | train: 705 | val: 50
GPU: True


In [21]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data="/content/dentex_yolo/dentex.yaml",
    epochs=100,
    patience=20,
    imgsz=1280,
    batch=8,
    seed=0,
    workers=2,
    project="/content/runs",
    name="baseline_v2",
)

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dentex_yolo/dentex.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_v2, nbs=64, nms=None, opset=None, optimize=Fals

In [23]:
# 1. استنساخ الـ repo
!git clone https://github.com/USERNAME/REPO.git
%cd REPO

# 2. إعداد اسمك وإيميلك
!git config --global user.name "اسمك"
!git config --global user.email "your_email@example.com"

# 3. بعد ما تضيفي أو تعدلي ملفات
!git add .
!git commit -m "Add my work"

# 4. الرفع (محتاجة Personal Access Token)
!git push https://TOKEN@github.com/USERNAME/REPO.git

Cloning into 'REPO'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'REPO'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [22]:
import shutil
from google.colab import files
shutil.make_archive("/content/baseline_v2", "zip", "/content/runs/baseline_v2")
files.download("/content/baseline_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>